In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import date2num, DateFormatter
import matplotlib.dates as mdates
import numpy as np
import glob
import datetime
import pytz
import warnings; warnings.simplefilter('ignore')
%matplotlib inline
n = 20
colors = plt.cm.turbo(np.linspace(0,1,n))
from datetime import timedelta
import xarray as xr
import metpy.calc as calc
from metpy.units import units
style = "/Users/cneumaie/OneDrive - Colostate/Styles/christine-paperlight.mpstyle"
import datetime
plt.style.use(style)
import matplotlib
matplotlib.rc('xtick', labelsize=32) 
matplotlib.rc('ytick', labelsize=32) 

Duplicate key in file '/Users/cneumaie/OneDrive - Colostate/Styles/christine-paperlight.mpstyle', line 15 ("xtick.color: 'k'")
Duplicate key in file '/Users/cneumaie/OneDrive - Colostate/Styles/christine-paperlight.mpstyle', line 16 ("ytick.color: 'k'")
Duplicate key in file '/Users/cneumaie/OneDrive - Colostate/Styles/christine-paperlight.mpstyle', line 27 ('axes.edgecolor:222222')


# This script is used to create a table that assigns each flight values for each variable claculated

In [2]:
dataPath = (
    "/Users/cneumaie/OneDrive - Colostate/Research/Drone_Profiles/00_Data/"
)

figPath = "/Users/cneumaie/OneDrive - Colostate/Research/Drone_Profiles/02_Figures/"
# CP_log = pd.read_csv(f'{dataPath}CP_database_BACS2_v13_enviru_thetae.csv')
CP_log = pd.read_csv(f'{dataPath}CP_database_BACS2_v14_enviru_thetae.csv')
profile_log_orig = pd.read_csv(f"{dataPath}prof_log_v16_airdata_id_v3_kmeans_clusters_v13_combo.csv")
profile_log_clusters = pd.read_csv(f"{dataPath}/prof_log_v16_airdata_id_v3_kmeans_clusters_v15_CPpairs.csv")
CP_log["Passage Time"] = pd.to_datetime(CP_log["Passage Time"])
profile_log_clusters["starttime"] = pd.to_datetime(profile_log_clusters["starttime"])

In [3]:
def get_times(FilePath):
    '''Converts surface station time data to datetime format and returns a pandas
    dataframe'''
    
    sfc_file = xr.open_dataset(glob.glob(FilePath+'BACS2_QC*.nc')[0])
    sfc_file
    df = sfc_file.to_dataframe()
    
    date_init = datetime.datetime(2022,1,1)   ##file is in format of seconds since Jan 1 2022
    ###Now we will convert the time to datetime function using a for loop
    datetime_list = []
    for i in range(len(sfc_file.time.values)):
        datetime_list.append(date_init+timedelta(minutes=int(sfc_file['time'][i].values))+timedelta(hours=6))###convert to UTC
    df['datetime'] = datetime_list
    ds = df.set_index('datetime') ###change the index to the datetime variable
    
    return ds

In [4]:
###load surface station data
sfc_filePath = '/Users/cneumaie/Downloads/'
sfc_df = get_times(sfc_filePath)
press = (sfc_df.pressure.values)*units.hPa
temp = (sfc_df.temperature.values)*units.degC
rh = (sfc_df.rh.values)/100
temp_K = (sfc_df.temperature.values + 273.15)*units.K
mixing_ratio = calc.mixing_ratio_from_relative_humidity(press, temp_K, rh)
t_d = calc.dewpoint_from_relative_humidity(temp,rh)
theta_e = calc.equivalent_potential_temperature(press,temp,t_d)
sfc_df["MR"] = mixing_ratio*1000
sfc_df["theta_e"] = theta_e

In [5]:
soil_rain_filePath = '/Users/cneumaie/Downloads/BACS_II_Met_data_v1.0.csv'

In [6]:
soil_rain_file = pd.read_csv(soil_rain_filePath)
soil_rain_file["time_UTC"] = pd.to_datetime(soil_rain_file.Mid_date_time_UTC_6) + pd.Timedelta(hours = 6)
soil_rain_file = soil_rain_file.set_index( soil_rain_file.time_UTC)

In [7]:
soil_rain_file

,Start_date_time_UTC_6,End_date_time_UTC_6,Mid_date_time_UTC_6,Temp_C,Temp_High_C,Temp_Low_C,RH_prctg,Dew_point_C,Wind_speed_m_s,Wind_speed_high_m_s,Wind_dir,Rainfall_mm,Rain_rate_mm_hr,time_UTC
time_UTC,,,,,,,,,,,,,,
2023-05-23 06:01:00,5/23/2023 0:01,5/23/2023 0:02,5/23/2023 0:01,12.9,12.9,12.9,82,9.9,0.4,0.4,ESE,0.0,0.0,2023-05-23 06:01:00
2023-05-23 06:02:00,5/23/2023 0:02,5/23/2023 0:03,5/23/2023 0:02,12.9,12.9,12.9,83,10.1,0.4,0.4,ESE,0.0,0.0,2023-05-23 06:02:00
2023-05-23 06:03:00,5/23/2023 0:03,5/23/2023 0:04,5/23/2023 0:03,12.9,12.9,12.9,82,9.9,0.4,0.4,ESE,0.0,0.0,2023-05-23 06:03:00
2023-05-23 06:04:00,5/23/2023 0:04,5/23/2023 0:05,5/23/2023 0:04,12.8,12.9,12.8,83,10.0,0.0,0.4,ESE,0.0,0.0,2023-05-23 06:04:00
2023-05-23 06:05:00,5/23/2023 0:05,5/23/2023 0:06,5/23/2023 0:05,12.8,12.8,12.8,82,9.8,0.0,0.0,-9999,0.0,0.0,2023-05-23 06:05:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-06-26 15:04:00,6/26/2023 9:04,6/26/2023 9:05,6/26/2023 9:04,19.6,19.6,19.5,73,14.6,0.4,0.9,SE,0.0,0.0,2023-06-26 15:04:00
2023-06-26 15:05:00,6/26/2023 9:05,6/26/2023 9:06,6/26/2023 9:05,19.6,19.6,19.5,73,14.6,0.4,1.8,E,0.0,0.0,2023-06-26 15:05:00
2023-06-26 15:06:00,6/26/2023 9:06,6/26/2023 9:07,6/26/2023 9:06,19.6,19.6,19.6,73,14.6,1.8,1.8,E,0.0,0.0,2023-06-26 15:06:00


In [8]:
soil_rainfall = soil_rain_file.Rainfall_mm ## rainfall over the 1 minute
soil_rainfall_rate = soil_rain_file.Rain_rate_mm_hr ### rainfall rate

In [9]:
sfc_df

,iop,temperature,rh,pressure,wspeed,wdir,u,v,MR,theta_e
datetime,,,,,,,,,,
2023-05-19 19:28:00,NaN,13.43250,71.50000,841.4002,2.510000,163.6681,-0.705815,2.408719,8.244495,325.720998
2023-05-19 19:29:00,NaN,13.39667,71.83334,841.3748,2.933334,209.1308,1.427962,2.562299,8.264115,325.738891
2023-05-19 19:30:00,NaN,13.08834,71.58334,841.3996,3.496667,213.5098,1.930439,2.915491,8.068523,324.813279
2023-05-19 19:31:00,NaN,13.08000,72.36667,841.3880,3.848334,221.4144,2.545674,2.886038,8.153585,325.050294
2023-05-19 19:32:00,NaN,12.69167,71.13333,841.3996,3.053333,227.6563,2.256772,2.056653,7.809093,323.605963
...,...,...,...,...,...,...,...,...,...,...
2023-06-27 19:02:00,NaN,31.08665,23.18333,826.9800,5.433332,230.4973,4.192330,3.456222,7.979059,347.172230
2023-06-27 19:03:00,NaN,30.83001,22.08333,826.9549,6.241667,231.2016,4.864477,3.910916,7.484209,345.318360
2023-06-27 19:04:00,NaN,30.56833,21.48333,826.8997,7.348333,231.4995,5.750825,4.574495,7.169537,344.032105


In [10]:
# profile_log_CP

In [11]:
profile_log_CP = profile_log_clusters[(profile_log_clusters.Condition == "CP") | (profile_log_clusters.Condition == "post-CP")]
profile_log_CP_orig =  profile_log_orig[(profile_log_orig.Condition == "CP") | (profile_log_orig.Condition == "post-CP")]
profile_log_CP["3cluster_minmax_CPonly"] = profile_log_CP_orig["3cluster_minmax_CPonly"]
profile_log_CP["4cluster_minmax_CPonly"] = profile_log_CP_orig["4cluster_minmax_CPonly"]

In [12]:
profile_log_CP

,Unnamed: 0.1,Unnamed: 0,index,Date,IOP,Type,Flight,profid,starttime,endtime,...,Unaspirated,count,starttime_ad,Condition,3cluster_minmax_CPpairs,4cluster_minmax_CPpairs,pre-CP_Tdiff,pre-CP_MRdiff,3cluster_minmax_CPonly,4cluster_minmax_CPonly
28,28,28,0,2023-05-27,17,ascending,3,0.0,2023-05-27 23:22:26,2023-05-27 23:24:13,...,X06,109.0,2023-05-27 17:22:00,post-CP,NaN,NaN,NaN,NaN,1.0,1.0
29,29,29,1,2023-05-27,17,ascending,3,1.0,2023-05-27 23:26:22,2023-05-27 23:28:06,...,X06,105.0,2023-05-27 17:22:00,post-CP,NaN,NaN,NaN,NaN,1.0,1.0
30,30,30,0,2023-05-27,17,descending,3,0.0,2023-05-27 23:24:25,2023-05-27 23:26:11,...,X06,107.0,2023-05-27 17:22:00,post-CP,NaN,NaN,NaN,NaN,1.0,1.0
31,31,31,1,2023-05-27,17,descending,3,1.0,2023-05-27 23:28:17,2023-05-27 23:30:10,...,X06,114.0,2023-05-27 17:22:00,post-CP,NaN,NaN,NaN,NaN,1.0,1.0
48,48,48,0,2023-05-30,18,ascending,3,0.0,2023-05-30 22:57:27,2023-05-30 22:59:19,...,X06,113.0,2023-05-30 16:56:00,CP,1.0,1.0,0.00000,0.0,0.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
371,371,371,2,2023-06-21,27,ascending,2,2.0,2023-06-21 22:12:24,2023-06-21 22:14:13,...,X11,111.0,2023-06-21 16:04:00,CP,2.0,0.0,-4.69451,-1.9,2.0,2.0
372,372,372,3,2023-06-21,27,ascending,2,3.0,2023-06-21 22:16:23,2023-06-21 22:18:48,...,X11,146.0,2023-06-21 16:04:00,CP,NaN,NaN,NaN,NaN,NaN,NaN
373,373,373,0,2023-06-21,27,descending,2,0.0,2023-06-21 22:06:22,2023-06-21 22:08:08,...,X11,109.0,2023-06-21 16:04:00,CP,2.0,0.0,-4.69451,-1.9,2.0,2.0
374,374,374,1,2023-06-21,27,descending,2,1.0,2023-06-21 22:10:23,2023-06-21 22:12:12,...,X11,110.0,2023-06-21 16:04:00,CP,2.0,0.0,-4.69451,-1.9,2.0,2.0


In [13]:
# def find_closest_datetime(datetime_list, target_datetime):
#     """
#     Finds the datetime in a list that is closest to a target datetime.

#     Args:
#         datetime_list: A list of datetime objects.
#         target_datetime: The datetime object to find the closest to.

#     Returns:
#         The datetime object in the list that is closest to the target datetime, or None if the list is empty.
#     """
#     # if not datetime_list:
#     #     return None

#     closest_datetime = min(datetime_list, key=lambda dt: abs(dt - target_datetime))
#     return closest_datetime


def find_closest_datetime(datetime_list, target_datetime, mode):
    """
    Finds the closest datetime in a list that is before or equal to a target datetime.

    Args:
        datetime_list: A list of datetime objects.
        target_datetime: The datetime object to find the closest to.

    Returns:
        The closest past datetime object in the list, or None if no valid datetime exists.
    """
    
    if mode == "post-CP":
        past_datetimes = [dt for dt in datetime_list if dt <= target_datetime]
        
        if not past_datetimes:
            return None  # Return None if no past datetime exists
        closest_datetime = max(past_datetimes) # Max will get the closest past datetime
    else:
        closest_datetime = min(datetime_list, key=lambda dt: abs(dt - target_datetime))

    return closest_datetime  

In [14]:
deltat_list = []
CP_id_list = []
dT_list = []
soil_rain_list = []
soil_rainrate_list = []
dMR_list = []
dte_list = []
ps_list = []
onset_list = []
last_CP_time_list = []
dT_leg_list = []
dMR_leg_list = []
dte_leg_list = []
dspeed_leg_list = []
duv_list = []
dbz_list = []
lcl_list = []
speed_list = []
distance_list = []
depth_list = []
pw_list = []
lr_list = []
vpd_list = []
lr_350_list = []
lr_lcl_list = []
for i in range(len(profile_log_CP)):
    temp_CP_log = profile_log_CP.iloc[i]
    temp_database = CP_log[CP_log.IOP == temp_CP_log.IOP]
    temp_CP_log["starttime_rnd"] = temp_CP_log["starttime"].round(freq="min")
    temp_sfc = sfc_df[sfc_df.index==temp_CP_log["starttime_rnd"]]
    T = temp_sfc.temperature.values
    MR = temp_sfc.MR.values
    te = temp_sfc.theta_e.values
    wspeed = temp_sfc.wspeed.values
    u = temp_sfc.u.values
    v = temp_sfc.v.values
    # print(temp_sfc)
    if len(temp_database) == 1:
        onset_time = temp_database.iloc[0]["Passage Time"]
        CP_id = temp_database.iloc[0]["CP ID"]
        dT = temp_database.iloc[0]["min dT(C)"]
        dMR = temp_database.iloc[0]["max dMR(g/kg)"]
        dte = temp_database.iloc[0]["min dte (K)"]
        ps = temp_database.iloc[0]["PS"]
        dbz = temp_database.iloc[0]["Max dBz"]
        lcl = temp_database.iloc[0]["LCL"]
        pw = temp_database.iloc[0]["PW"]
        speed = temp_database.iloc[0]["CP_speed2"]
        depth = temp_database.iloc[0]["CP_depth"]
        lr = temp_database.iloc[0]["Sfc-3km_LR"]
        vpd = temp_database.iloc[0]["VPD"]
        lr_350 = temp_database.iloc[0]["Sfc-350m_LR"]
        lr_lcl = temp_database.iloc[0]["Sfc-LCL_LR"]
        
        envir_T = temp_database.iloc[0]["envir_T"]
        envir_MR = temp_database.iloc[0]["envir_MR"]
        envir_te = temp_database.iloc[0]["envir_te"]
        envir_wspeed = temp_database.iloc[0]["envir_wspeed"]
        envir_u = temp_database.iloc[0]["envir_u"]
        envir_v = temp_database.iloc[0]["envir_v"]
        try:
            dT_leg_list.append(-np.round(envir_T-T,1)[0])
        except:
            dT_leg_list.append(np.nan)
        try:
            dMR_leg_list.append(-np.round(envir_MR-MR,2)[0])
        except:
            dMR_leg_list.append(np.nan)
        try:
            dte_leg_list.append(-np.round(envir_te-te,2)[0])
        except:
            dte_leg_list.append(np.nan)
        try:
            dspeed_leg_list.append(-np.round(envir_wspeed-wspeed,2)[0])
        except:
            dspeed_leg_list.append(np.nan)
        try:
            duv = np.sqrt((u-envir_u)**2 + (v-envir_v)**2)
            duv_list.append(np.round(duv,2)[0])
        except:
            duv_list.append(np.nan)
        
        dtime = temp_CP_log.starttime-onset_time
        deltat_list.append(dtime)
        distance = speed*(pd.to_timedelta(dtime).total_seconds()/60)
        speed_list.append(speed)
        distance_list.append(distance)
        depth_list.append(depth)
        dT_list.append(dT)
        dMR_list.append(dMR)
        dte_list.append(dte)
        CP_id_list.append(CP_id)
        onset_list.append(onset_time)
        ps_list.append(ps)
        dbz_list.append(dbz)
        lcl_list.append(lcl)
        pw_list.append(pw)
        lr_list.append(lr)
        vpd_list.append(vpd)
        lr_350_list.append(lr_350)
        lr_lcl_list.append(lr_lcl)
        if i ==0:
            last_CP_time_list.append(pd.Timedelta(days=1))
        else:
            if CP_id == CP_id_list[-2]:
                last_CP_time_list.append(last_CP_time_list[-1])
            else:
                last_CP_time = onset_list[-2]
                CP_time_diff = onset_time-last_CP_time
                if CP_time_diff < pd.Timedelta(minutes=0):
                    print(onset_time, last_CP_time)
                last_CP_time_list.append(CP_time_diff)
                                    
        
    else:
        passage_times = temp_database["Passage Time"].values
        closest_time = find_closest_datetime(passage_times, temp_CP_log.starttime, mode = temp_CP_log.Condition)
        if temp_CP_log.starttime < closest_time and temp_CP_log.Condition == "post-CP":
            onset_index = np.where(temp_database["Passage Time"]==closest_time-1)
        else:
            onset_ind = np.where(temp_database["Passage Time"]==closest_time)
        onset_time = temp_database.iloc[onset_ind]["Passage Time"]
        CP_id = temp_database.iloc[onset_ind]["CP ID"].values
        dT = temp_database.iloc[onset_ind]["min dT(C)"].values
        dMR = temp_database.iloc[onset_ind]["max dMR(g/kg)"].values
        dte = temp_database.iloc[onset_ind]["min dte (K)"].values
        ps = temp_database.iloc[onset_ind]["PS"].values[0]
        dbz = temp_database.iloc[onset_ind]["Max dBz"].values[0]
        lcl = temp_database.iloc[onset_ind]["LCL"].values[0]
        pw = temp_database.iloc[onset_ind]["PW"].values[0]
        speed = temp_database.iloc[onset_ind]["CP_speed2"].values[0]
        depth = temp_database.iloc[onset_ind]["CP_depth"].values[0]
        lr = temp_database.iloc[onset_ind]["Sfc-3km_LR"].values[0]
        vpd = temp_database.iloc[onset_ind]["VPD"].values[0]
        lr_350 = temp_database.iloc[onset_ind]["Sfc-350m_LR"].values[0]
        lr_lcl = temp_database.iloc[onset_ind]["Sfc-LCL_LR"].values[0]
        dtime = temp_CP_log.starttime-onset_time
        # if pd.Timedelta(dtime.values[0].astype(int), unit = 'ns') < pd.Timedelta(days=-1,hours=23,minutes = 20):
            # print(temp_CP_log)
            # print(temp_database)
            # print('time', pd.Timedelta(dtime.values[0].astype(int), unit = 'ns'))
            # print(onset_time)
        envir_T = temp_database.iloc[onset_ind]["envir_T"].values
        envir_MR = temp_database.iloc[onset_ind]["envir_MR"].values
        envir_dte = temp_database.iloc[onset_ind]["envir_te"].values
        envir_wspeed = temp_database.iloc[onset_ind]["envir_wspeed"].values
        envir_u = temp_database.iloc[onset_ind]["envir_u"].values
        envir_v = temp_database.iloc[onset_ind]["envir_v"].values
        try:
            dT_leg_list.append(-np.round(envir_T-T,1)[0])
        except:
            print(T)
            dT_leg_list.append(np.nan)
        try:
            dMR_leg_list.append(-np.round(envir_MR-MR,2)[0])
        except:
            dMR_leg_list.append(np.nan)
        try:
            dte_leg_list.append(-np.round(envir_te-te,2)[0])
        except:
            dte_leg_list.append(np.nan)
        try:
            dspeed_leg_list.append(-np.round(envir_wspeed-wspeed,2)[0])
        except:
            dspeed_leg_list.append(np.nan)
        try:
            duv = np.sqrt((u-envir_u)**2 + (v-envir_v)**2)
            duv_list.append(np.round(duv,2)[0])
        except:
            print(duv)
            duv_list.append(np.nan)
        deltat_list.append(pd.Timedelta(dtime.values[0].astype(int), unit = 'ns'))
        distance = speed*(pd.to_timedelta(dtime.values[0]).total_seconds())
        speed_list.append(speed)
        distance_list.append(distance)
        depth_list.append(depth)
        dT_list.append(dT[0])
        dMR_list.append(dMR[0])
        dte_list.append(dte[0])
        CP_id_list.append(CP_id[0])
        onset_list.append(pd.Timestamp(onset_time.values[0]))
        ps_list.append(ps)
        dbz_list.append(dbz)
        lcl_list.append(lcl)
        pw_list.append(pw)
        lr_list.append(lr)
        vpd_list.append(vpd)
        lr_350_list.append(lr_350)
        lr_lcl_list.append(lr_lcl)
        if i ==0:
            last_CP_time_list.append(pd.Timedelta(days=1))
        else:
            if CP_id == CP_id_list[-2]:
                last_CP_time_list.append(last_CP_time_list[-1])
            elif CP_id < CP_id_list[-2]:
                cp_ind = np.where(CP_id_list == CP_id)[0][0]
                last_CP_time_list.append(last_CP_time_list[cp_ind])
            else:
                last_CP_time = onset_list[-2]
                CP_time_diff = onset_list[-1]-last_CP_time
                if CP_time_diff < pd.Timedelta(minutes=0):
                    print(onset_time, last_CP_time, CP_time_diff)
                last_CP_time_list.append(CP_time_diff)
        
        
    
    

In [15]:
# duv_list

In [16]:
# pw_list

In [17]:
# deltat_list

In [18]:
# temp_database.iloc[0]["envir_T"]

In [19]:
# dT_leg_list

In [20]:
# last_CP_time_list

In [21]:
profile_log_CP["CP_id"] = CP_id_list
profile_log_CP["onset_time"] = onset_list
profile_log_CP["delta_time"] = deltat_list
profile_log_CP["min dT"] = dT_list
profile_log_CP["max dMR"] = dMR_list
profile_log_CP["min dte"] = dte_list
profile_log_CP["PS"] = ps_list
profile_log_CP["prev_CP_time"] = last_CP_time_list
profile_log_CP["dT"] = dT_leg_list
profile_log_CP["dMR"] = dMR_leg_list
profile_log_CP["dte"] = dte_leg_list
profile_log_CP["dwspeed"] = dspeed_leg_list
profile_log_CP["duv"] = duv_list
profile_log_CP["dBZ"] = dbz_list
profile_log_CP["LCL"] = lcl_list
profile_log_CP["PW"] = pw_list
profile_log_CP["Sfc-3km_LR"] = lr_list
profile_log_CP["VPD"] = vpd_list
profile_log_CP["Sfc-350m_LR"] = lr_350_list
profile_log_CP["Sfc-LCL_LR"] = lr_lcl_list
profile_log_CP["CP_speed"] = speed_list
profile_log_CP["CP_depth"] = depth_list
profile_log_CP["CP_distance"] = distance_list



In [22]:
# profile_log_CP.to_csv(f"{dataPath}CP_prof_log_BACS2_v18_test.csv")
profile_log_CP.to_csv(f"{dataPath}CP_prof_log_BACS2_v19_test.csv")

In [232]:
# profile_log_CP[(profile_log_CP["IOP"]==23) & (profile_log_CP["Condition"]=="post-CP")]

In [56]:
profile_log_CP.keys()

Index(['Unnamed: 0.1', 'Unnamed: 0', 'index', 'Date', 'IOP', 'Type', 'Flight',
       'profid', 'starttime', 'endtime', 'Aspirated', 'Unaspirated', 'count',
       'starttime_ad', 'Condition', '3cluster_minmax_CPpairs',
       '4cluster_minmax_CPpairs', 'pre-CP_Tdiff', 'pre-CP_MRdiff',
       '3cluster_minmax_CPonly', '4cluster_minmax_CPonly', 'CP_id',
       'onset_time', 'delta_time', 'min dT', 'max dMR', 'PS', 'prev_CP_time',
       'dT', 'dMR', 'dwspeed', 'duv', 'dBZ', 'LCL', 'PW', 'Sfc-3km_LR', 'VPD',
       'Sfc-350m_LR', 'Sfc-LCL_LR'],
      dtype='object')

In [43]:
profile_log_CP

,Unnamed: 0.1,Unnamed: 0,index,Date,IOP,Type,Flight,profid,starttime,endtime,...,dte,dwspeed,duv,dBZ,LCL,PW,Sfc-3km_LR,VPD,Sfc-350m_LR,Sfc-LCL_LR
28,28,28,0,2023-05-27,17,ascending,3,0.0,2023-05-27 23:22:26,2023-05-27 23:24:13,...,-14.46,-0.54,11.84,65.0,1092.0,0.47,8.7,6.63,13.17,5.68
29,29,29,1,2023-05-27,17,ascending,3,1.0,2023-05-27 23:26:22,2023-05-27 23:28:06,...,-14.65,0.70,12.36,65.0,1092.0,0.47,8.7,6.63,13.17,5.68
30,30,30,0,2023-05-27,17,descending,3,0.0,2023-05-27 23:24:25,2023-05-27 23:26:11,...,-14.45,-0.96,10.92,65.0,1092.0,0.47,8.7,6.63,13.17,5.68
31,31,31,1,2023-05-27,17,descending,3,1.0,2023-05-27 23:28:17,2023-05-27 23:30:10,...,-14.46,-0.51,11.36,65.0,1092.0,0.47,8.7,6.63,13.17,5.68
48,48,48,0,2023-05-30,18,ascending,3,0.0,2023-05-30 22:57:27,2023-05-30 22:59:19,...,-7.21,-2.82,2.92,45.0,1850.0,0.49,9.6,10.71,15.36,9.11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
371,371,371,2,2023-06-21,27,ascending,2,2.0,2023-06-21 22:12:24,2023-06-21 22:14:13,...,-12.34,3.93,9.89,70.0,886.0,0.96,9.0,5.47,16.00,11.65
372,372,372,3,2023-06-21,27,ascending,2,3.0,2023-06-21 22:16:23,2023-06-21 22:18:48,...,-14.74,6.30,11.70,70.0,886.0,0.96,9.0,5.47,16.00,11.65
373,373,373,0,2023-06-21,27,descending,2,0.0,2023-06-21 22:06:22,2023-06-21 22:08:08,...,-5.59,0.08,5.87,70.0,886.0,0.96,9.0,5.47,16.00,11.65
374,374,374,1,2023-06-21,27,descending,2,1.0,2023-06-21 22:10:23,2023-06-21 22:12:12,...,-14.16,2.67,9.23,70.0,886.0,0.96,9.0,5.47,16.00,11.65
